# NB_{{DOM}}_Load · Phase 1 · Oracle EBS sample data → {{LH}}
Generates a deterministic Oracle EBS R12 extract for KDD (all tables, seed 42), keeps this domain's tables,
writes them as CSV to `Files/ebs_landing/` (the simulated EBS extract) and loads them unchanged into schema `ebs`
with `_load_ts` and `_source_file`. Suppliers, POs and invoices are fictional.

In [ ]:
%%configure -f
{"defaultLakehouse": {"name": "{{LH}}"}}

In [ ]:
DOMAIN = "{{DOM}}"          # PRC or AP
LANDING = "/lakehouse/default/Files/ebs_landing"

In [ ]:
import random, os, datetime as dt
import pandas as pd
random.seed(42)
D0 = dt.date(2026, 6, 1)
def d(days): return (D0 + dt.timedelta(days=days)).isoformat()
ORGS = [(101, "KDD Kuwait", "KWD"), (102, "KDD Saudi Arabia", "SAR"), (103, "KDD UAE", "AED")]
CUR = {o[0]: o[2] for o in ORGS}
T = {}
T["HR_OPERATING_UNITS"] = pd.DataFrame([dict(ORGANIZATION_ID=o, NAME=n, SHORT_CODE=n.split()[-1][:3].upper(), SET_OF_BOOKS_ID=2000 + i, DATE_FROM="2015-01-01") for i, (o, n, c) in enumerate(ORGS)])
cats = ["Dairy", "Ice cream", "Juices", "Culinary", "Raw materials", "Packaging"]
T["MTL_CATEGORIES_B"] = pd.DataFrame([dict(CATEGORY_ID=10 + i, SEGMENT1=c, DESCRIPTION=f"{c} purchasing category", ENABLED_FLAG="Y") for i, c in enumerate(cats)])
items = [("Skimmed milk powder 25kg", "KG", "Dairy", 1.05), ("Full cream milk powder 25kg", "KG", "Dairy", 1.22), ("Raw cow milk", "L", "Dairy", 0.16),
         ("Cream 35% fat", "L", "Dairy", 0.95), ("Unsalted butter 25kg", "KG", "Dairy", 1.60), ("Cheese cultures", "EA", "Dairy", 12.50), ("Microbial rennet", "L", "Dairy", 8.40),
         ("Mango concentrate", "KG", "Juices", 0.62), ("Orange concentrate", "KG", "Juices", 0.55), ("Guava puree", "KG", "Juices", 0.48),
         ("Cocoa powder 10/12", "KG", "Ice cream", 2.10), ("Vanilla flavour", "KG", "Ice cream", 9.80), ("Stabiliser blend E407", "KG", "Ice cream", 3.30),
         ("Ice cream cup 120ml", "EA", "Ice cream", 0.012), ("Wooden stick", "EA", "Ice cream", 0.003),
         ("Sunflower oil", "L", "Culinary", 0.42), ("Tomato paste 28%", "KG", "Culinary", 0.38), ("Corn starch", "KG", "Culinary", 0.21), ("Salt refined", "KG", "Culinary", 0.04),
         ("White sugar ICUMSA 45", "KG", "Raw materials", 0.17), ("CIP caustic 50%", "L", "Raw materials", 0.11), ("Lab reagent kit", "EA", "Raw materials", 18.00),
         ("Aseptic carton 1L", "EA", "Packaging", 0.045), ("Aseptic carton 250ml", "EA", "Packaging", 0.021), ("PET bottle 500ml", "EA", "Packaging", 0.032),
         ("HDPE cap 38mm", "EA", "Packaging", 0.006), ("Shrink film", "KG", "Packaging", 0.55), ("Corrugated case", "EA", "Packaging", 0.090),
         ("Self-adhesive label", "EA", "Packaging", 0.004), ("Pallet stretch wrap", "KG", "Packaging", 0.60)]
catid = {c: 10 + i for i, c in enumerate(cats)}
T["MTL_SYSTEM_ITEMS_B"] = pd.DataFrame([dict(INVENTORY_ITEM_ID=5000 + i, ORGANIZATION_ID=100, SEGMENT1=f"KDD-{5000+i}", DESCRIPTION=n, PRIMARY_UOM_CODE=u,
                                             CATEGORY_ID=catid[c], LIST_PRICE_PER_UNIT=p) for i, (n, u, c, p) in enumerate(items)])
sups = [("Gulf Dairy Ingredients Co", "KW", "Dairy"), ("Nordic Milk Powders A/S", "DK", "Dairy"), ("Al Wafra Packaging WLL", "KW", "Packaging"),
        ("Emirates Fruit Concentrates LLC", "AE", "Juices"), ("Riyadh Sugar Trading Est", "SA", "Raw materials"), ("Euro Aseptic Packaging GmbH", "DE", "Packaging"),
        ("Kuwait Cold Chain Logistics", "KW", "Raw materials"), ("Levant Cocoa Traders", "JO", "Ice cream"), ("Indo Spices & Flavours Pvt Ltd", "IN", "Ice cream"),
        ("Arabian PET Solutions", "SA", "Packaging"), ("Hamburg Cultures & Enzymes GmbH", "DE", "Dairy"), ("Jeddah Edible Oils Co", "SA", "Culinary"),
        ("Bahrain Labels & Print WLL", "BH", "Packaging"), ("Qatar Corrugated Boxes", "QA", "Packaging"), ("Desert Chemicals Kuwait", "KW", "Raw materials")]
T["AP_SUPPLIERS"] = pd.DataFrame([dict(VENDOR_ID=3000 + i, SEGMENT1=f"V{3000+i}", VENDOR_NAME=n, VENDOR_TYPE_LOOKUP_CODE="VENDOR",
                                       VAT_REGISTRATION_NUM=f"{cc}{random.randint(10**9, 10**10-1)}", ATTRIBUTE1=c, ENABLED_FLAG="Y",
                                       START_DATE_ACTIVE="2020-01-01", END_DATE_ACTIVE="") for i, (n, cc, c) in enumerate(sups)])
T["AP_TERMS"] = pd.DataFrame([dict(TERM_ID=1, NAME="Immediate", DUE_DAYS=0), dict(TERM_ID=2, NAME="Net 30", DUE_DAYS=30),
                              dict(TERM_ID=3, NAME="Net 45", DUE_DAYS=45), dict(TERM_ID=4, NAME="Net 60", DUE_DAYS=60)])
city = {"KW": "Kuwait City", "DK": "Aarhus", "AE": "Dubai", "SA": "Riyadh", "DE": "Hamburg", "JO": "Amman", "IN": "Mumbai", "BH": "Manama", "QA": "Doha"}
sites = []; sid = 7000
for i, (n, cc, c) in enumerate(sups):
    for o in ([101] + ([102] if i % 2 == 0 else []) + ([103] if i % 3 == 0 else []))[: 1 + (i % 3 > 0)]:
        sites.append(dict(VENDOR_SITE_ID=sid, VENDOR_ID=3000 + i, ORG_ID=o, VENDOR_SITE_CODE=f"{city[cc][:6].upper()}-{o}", COUNTRY=cc, CITY=city[cc], TERMS_ID=random.choice([2, 2, 3, 4]))); sid += 1
while len(sites) < 25:
    i = random.randrange(15); o = random.choice([102, 103])
    if not any(s["VENDOR_ID"] == 3000 + i and s["ORG_ID"] == o for s in sites):
        sites.append(dict(VENDOR_SITE_ID=sid, VENDOR_ID=3000 + i, ORG_ID=o, VENDOR_SITE_CODE=f"{city[sups[i][1]][:6].upper()}-{o}", COUNTRY=sups[i][1], CITY=city[sups[i][1]], TERMS_ID=2)); sid += 1
T["AP_SUPPLIER_SITES_ALL"] = pd.DataFrame(sites)
cur_by_cc = {"KW": "KWD", "SA": "SAR", "AE": "AED", "DK": "EUR", "DE": "EUR", "JO": "USD", "IN": "USD", "BH": "USD", "QA": "USD"}
banks = {"KWD": "National Bank of Kuwait", "SAR": "Saudi National Bank", "AED": "Emirates NBD", "EUR": "Deutsche Bank", "USD": "Citibank"}
def iban(cc):
    return cc[:2].replace("DK", "DK").upper() + f"{random.randint(10,99)}" + "".join(random.choice("ABCDEFGHIJKLMNOPQRSTUVWXYZ") for _ in range(4)) + "".join(str(random.randint(0, 9)) for _ in range(18))
T["IBY_EXT_BANK_ACCOUNTS"] = pd.DataFrame([dict(EXT_BANK_ACCOUNT_ID=9000 + i, VENDOR_ID=3000 + i, BANK_NAME=banks[cur_by_cc[cc]], BANK_ACCOUNT_NUM=str(random.randint(10**11, 10**12 - 1)),
                                                IBAN=iban(cc), CURRENCY_CODE=cur_by_cc[cc]) for i, (n, cc, c) in enumerate(sups)])
# requisitions
reqh, reql = [], []; rl = 1
for i in range(40):
    o = [101] * 17 + [102] * 13 + [103] * 10; o = o[i]
    reqh.append(dict(REQUISITION_HEADER_ID=8000 + i, SEGMENT1=f"RQ{26000+i}", ORG_ID=o, PREPARER_ID=200 + i % 7, AUTHORIZATION_STATUS=random.choice(["APPROVED"] * 4 + ["IN PROCESS"]), CREATION_DATE=d(i * 3)))
for n in range(90):
    h = reqh[n % 40]; it = random.randrange(30)
    reql.append(dict(REQUISITION_LINE_ID=81000 + n, REQUISITION_HEADER_ID=h["REQUISITION_HEADER_ID"], LINE_NUM=1 + n // 40, ITEM_ID=5000 + it,
                     QUANTITY=random.choice([500, 1000, 2000, 5000, 10000]), UNIT_PRICE=items[it][3], NEED_BY_DATE=d(n % 40 * 3 + 21), DESTINATION_ORGANIZATION_ID=h["ORG_ID"]))
T["PO_REQUISITION_HEADERS_ALL"] = pd.DataFrame(reqh); T["PO_REQUISITION_LINES_ALL"] = pd.DataFrame(reql)
# purchase orders
poh, pol, pod = [], [], []
orgseq = [101] * 25 + [102] * 20 + [103] * 15
for i in range(60):
    o = orgseq[i]; site = random.choice([s for s in sites if s["ORG_ID"] == o])
    poh.append(dict(PO_HEADER_ID=10000 + i, SEGMENT1=f"PO{4500+i}", ORG_ID=o, VENDOR_ID=site["VENDOR_ID"], VENDOR_SITE_ID=site["VENDOR_SITE_ID"], CURRENCY_CODE=CUR[o],
                    CREATION_DATE=d(i * 2 + 1), AUTHORIZATION_STATUS=random.choice(["APPROVED"] * 8 + ["IN PROCESS", "INCOMPLETE"]),
                    CLOSED_CODE="CLOSED" if i < 30 else "OPEN", TYPE_LOOKUP_CODE="STANDARD", AGENT_ID=300 + i % 5))
fx = {"KWD": 1.0, "SAR": 0.0819, "AED": 0.0836}
for n in range(180):
    h = poh[n % 60]; it = random.randrange(30); local_price = round(items[it][3] / fx[h["CURRENCY_CODE"]] * random.uniform(0.9, 1.1), 4)
    pol.append(dict(PO_LINE_ID=20000 + n, PO_HEADER_ID=h["PO_HEADER_ID"], LINE_NUM=1 + n // 60, ITEM_ID=5000 + it, CATEGORY_ID=catid[items[it][2]], ITEM_DESCRIPTION=items[it][0],
                    UNIT_MEAS_LOOKUP_CODE=items[it][1], QUANTITY=random.choice([500, 1000, 2500, 5000, 10000, 20000]), UNIT_PRICE=local_price,
                    NEED_BY_DATE=(dt.date.fromisoformat(h["CREATION_DATE"]) + dt.timedelta(days=random.choice([14, 21, 30]))).isoformat(), ORG_ID=h["ORG_ID"]))
    pod.append(dict(PO_DISTRIBUTION_ID=30000 + n, PO_HEADER_ID=h["PO_HEADER_ID"], PO_LINE_ID=20000 + n, CODE_COMBINATION_ID=14000 + catid[items[it][2]],
                    QUANTITY_ORDERED=pol[-1]["QUANTITY"], ORG_ID=h["ORG_ID"]))
T["PO_HEADERS_ALL"] = pd.DataFrame(poh); T["PO_LINES_ALL"] = pd.DataFrame(pol); T["PO_DISTRIBUTIONS_ALL"] = pd.DataFrame(pod)
# receipts
rsh, rtx = [], []
for i in range(50):
    h = poh[i]; rsh.append(dict(SHIPMENT_HEADER_ID=40000 + i, RECEIPT_NUM=f"RCV{7700+i}", VENDOR_ID=h["VENDOR_ID"], SHIP_TO_ORG_ID=h["ORG_ID"], CREATION_DATE=d(i * 2 + 18)))
for n in range(150):
    sh = rsh[n % 50]; ln = [l for l in pol if l["PO_HEADER_ID"] == 10000 + n % 50][n // 50 % 3]
    typ = "RECEIVE" if n < 100 else ("DELIVER" if n < 145 else "RETURN TO VENDOR")
    q = ln["QUANTITY"] if typ != "RETURN TO VENDOR" else round(ln["QUANTITY"] * 0.05)
    rtx.append(dict(TRANSACTION_ID=50000 + n, SHIPMENT_HEADER_ID=sh["SHIPMENT_HEADER_ID"], TRANSACTION_TYPE=typ, TRANSACTION_DATE=sh["CREATION_DATE"], QUANTITY=q,
                    PO_HEADER_ID=ln["PO_HEADER_ID"], PO_LINE_ID=ln["PO_LINE_ID"], ORGANIZATION_ID=sh["SHIP_TO_ORG_ID"]))
T["RCV_SHIPMENT_HEADERS"] = pd.DataFrame(rsh); T["RCV_TRANSACTIONS"] = pd.DataFrame(rtx)
# invoices: 2 per PO, lines map to that PO's lines
inv, invl, invd, sched = [], [], [], []
terms = {s["VENDOR_SITE_ID"]: s["TERMS_ID"] for s in sites}; due = {1: 0, 2: 30, 3: 45, 4: 60}
for i in range(120):
    h = poh[i % 60]; lines = [l for l in pol if l["PO_HEADER_ID"] == h["PO_HEADER_ID"]][: 2]
    idate = dt.date.fromisoformat(h["CREATION_DATE"]) + dt.timedelta(days=20 + (i // 60) * 15)
    amts = [round(l["QUANTITY"] * l["UNIT_PRICE"] * (0.5 if i < 60 else 0.5), 2) for l in lines]
    iid = 60000 + i; tid = terms[h["VENDOR_SITE_ID"]]
    inv.append(dict(INVOICE_ID=iid, INVOICE_NUM=f"INV-{h['VENDOR_ID']}-{i:04d}", VENDOR_ID=h["VENDOR_ID"], VENDOR_SITE_ID=h["VENDOR_SITE_ID"], ORG_ID=h["ORG_ID"],
                    INVOICE_DATE=idate.isoformat(), INVOICE_AMOUNT=round(sum(amts), 2), INVOICE_CURRENCY_CODE=h["CURRENCY_CODE"], TERMS_ID=tid,
                    APPROVAL_STATUS=random.choice(["APPROVED"] * 9 + ["NEEDS REAPPROVAL"]), PAYMENT_STATUS_FLAG="N"))
    for k, (l, a) in enumerate(zip(lines, amts)):
        invl.append(dict(INVOICE_ID=iid, LINE_NUMBER=k + 1, LINE_TYPE_LOOKUP_CODE="ITEM", AMOUNT=a, PO_HEADER_ID=l["PO_HEADER_ID"], PO_LINE_ID=l["PO_LINE_ID"], ORG_ID=h["ORG_ID"]))
        invd.append(dict(INVOICE_DISTRIBUTION_ID=61000 + len(invd), INVOICE_ID=iid, INVOICE_LINE_NUMBER=k + 1, AMOUNT=a, DIST_CODE_COMBINATION_ID=14000 + l["CATEGORY_ID"], ORG_ID=h["ORG_ID"]))
    sched.append(dict(INVOICE_ID=iid, PAYMENT_NUM=1, DUE_DATE=(idate + dt.timedelta(days=due[tid])).isoformat(), AMOUNT_REMAINING=inv[-1]["INVOICE_AMOUNT"], ORG_ID=h["ORG_ID"]))
# payments: 70 checks paying 90 invoices (first 90 invoices by date)
bank = {r["VENDOR_ID"]: r["EXT_BANK_ACCOUNT_ID"] for r in T["IBY_EXT_BANK_ACCOUNTS"].to_dict("records")}
paid = sorted(inv, key=lambda x: x["INVOICE_DATE"])[:90]; chk, ipay = [], []
groups = [paid[i:i + 1] for i in range(50)] + [paid[50 + 2 * j: 52 + 2 * j] for j in range(20)]
for c, g in enumerate(groups):
    cid = 70000 + c; total = round(sum(x["INVOICE_AMOUNT"] for x in g), 2); pdate = (dt.date.fromisoformat(g[-1]["INVOICE_DATE"]) + dt.timedelta(days=28)).isoformat()
    chk.append(dict(CHECK_ID=cid, CHECK_NUMBER=900100 + c, VENDOR_ID=g[0]["VENDOR_ID"], AMOUNT=total, CURRENCY_CODE=g[0]["INVOICE_CURRENCY_CODE"], CHECK_DATE=pdate,
                    BANK_ACCOUNT_NAME=f"KDD {g[0]['INVOICE_CURRENCY_CODE']} Operating", EXTERNAL_BANK_ACCOUNT_ID=bank[g[0]["VENDOR_ID"]], ORG_ID=g[0]["ORG_ID"], STATUS_LOOKUP_CODE="NEGOTIABLE"))
    for x in g:
        ipay.append(dict(INVOICE_PAYMENT_ID=71000 + len(ipay), INVOICE_ID=x["INVOICE_ID"], CHECK_ID=cid, AMOUNT=x["INVOICE_AMOUNT"], ACCOUNTING_DATE=pdate, ORG_ID=x["ORG_ID"]))
        x["PAYMENT_STATUS_FLAG"] = "Y"; next(s for s in sched if s["INVOICE_ID"] == x["INVOICE_ID"])["AMOUNT_REMAINING"] = 0
T["AP_INVOICES_ALL"] = pd.DataFrame(inv); T["AP_INVOICE_LINES_ALL"] = pd.DataFrame(invl); T["AP_INVOICE_DISTRIBUTIONS_ALL"] = pd.DataFrame(invd)
T["AP_PAYMENT_SCHEDULES_ALL"] = pd.DataFrame(sched); T["AP_CHECKS_ALL"] = pd.DataFrame(chk); T["AP_INVOICE_PAYMENTS_ALL"] = pd.DataFrame(ipay)
print({k: len(v) for k, v in T.items()})

In [ ]:
OWN = {"PRC": ["HR_OPERATING_UNITS", "MTL_CATEGORIES_B", "MTL_SYSTEM_ITEMS_B", "PO_REQUISITION_HEADERS_ALL", "PO_REQUISITION_LINES_ALL",
               "PO_HEADERS_ALL", "PO_LINES_ALL", "PO_DISTRIBUTIONS_ALL", "RCV_SHIPMENT_HEADERS", "RCV_TRANSACTIONS"],
       "AP": ["HR_OPERATING_UNITS", "AP_SUPPLIERS", "AP_SUPPLIER_SITES_ALL", "AP_TERMS", "IBY_EXT_BANK_ACCOUNTS", "AP_INVOICES_ALL", "AP_INVOICE_LINES_ALL",
              "AP_INVOICE_DISTRIBUTIONS_ALL", "AP_PAYMENT_SCHEDULES_ALL", "AP_CHECKS_ALL", "AP_INVOICE_PAYMENTS_ALL"]}[DOMAIN]
os.makedirs(LANDING, exist_ok=True)
for t in OWN:
    T[t].to_csv(f"{LANDING}/{t}.csv", index=False)
print("Landed", len(OWN), "files in Files/ebs_landing")

In [ ]:
from pyspark.sql import functions as F
spark.sql("CREATE SCHEMA IF NOT EXISTS ebs")
counts = {}
for t in OWN:
    df = (spark.read.option("header", True).option("inferSchema", False).csv(f"Files/ebs_landing/{t}.csv")
          .withColumn("_load_ts", F.current_timestamp()).withColumn("_source_file", F.lit(f"ebs_landing/{t}.csv")))
    df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"ebs.{t.lower()}")
    counts[t] = spark.table(f"ebs.{t.lower()}").count()
display(pd.DataFrame(sorted(counts.items()), columns=["table", "rows"]))